# Fine-tuning de DistilBERT para análisis de sentimiento en español

En la primera versión de este notebook entrenamos DistilBERT con 15 frases, pero el modelo no aprendió a distinguir las positivas de las negativas. ¿Fue por tener poca data? En parte sí, pero ese no era el problema principal:

| Problema | Por qué afecta | Solución |
|---|---|---|
| `distilbert-base-uncased` se preentrenó **solo en inglés** | No conoce las palabras en español: las parte en trozos sin sentido y no aprovecha lo que aprendió en el preentrenamiento | Partir de un DistilBERT preentrenado **en español** |
| 15 frases: 10 positivas y 5 negativas | Muy pocos ejemplos y clases desbalanceadas | 25 + 25 frases balanceadas y, después, reseñas reales |
| Evaluábamos con las mismas frases del entrenamiento | Mide cuánto memoriza, no cuánto generaliza | Frases de prueba que el modelo nunca ve |

Plan del notebook:

1. Ver por qué el modelo en inglés no sirve para español.
2. Comparar **modelo** (inglés o español) y **datos** (15 o 50 frases) para saber qué pesa más.
3. Comprobar el resultado con **reseñas reales de Amazon**, que son más difíciles que nuestras frases.
4. Entrenar con 2,000 reseñas reales, guardar el modelo y usarlo.

In [1]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, pipeline, set_seed)


SEED = 42
set_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"DEVICE: {DEVICE}")

MODELO_INGLES = "distilbert-base-uncased"                    # el que usábamos
MODELO_ESPANOL = "dccuchile/distilbert-base-spanish-uncased"  # misma arquitectura, preentrenado en español
ETIQUETAS = {0: "negativo", 1: "positivo"}

/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: mps


## 1. El modelo base importa más de lo que parece

Hacer fine-tuning es **transfer learning**, igual que con las CNN preentrenadas en ImageNet: el modelo ya sabe mucho del lenguaje y solo le enseñamos la tarea final. Pero eso funciona solo si el modelo base conoce el idioma de nuestros textos. Veamos cómo ve cada tokenizador una misma frase:

In [2]:
frase = "La atención fue pésima, no volveré nunca."

for nombre in [MODELO_INGLES, MODELO_ESPANOL]:
    tokens = AutoTokenizer.from_pretrained(nombre).tokenize(frase)
    print(f"{nombre} ({len(tokens)} tokens)\n  {tokens}\n")

distilbert-base-uncased (16 tokens)
  ['la', 'ate', '##nc', '##ion', 'fu', '##e', 'pe', '##si', '##ma', ',', 'no', 'vol', '##vere', 'nun', '##ca', '.']

dccuchile/distilbert-base-spanish-uncased (11 tokens)
  ['la', 'atención', 'fue', 'pé', '##si', '##ma', ',', 'no', 'volveré', 'nunca', '.']



El modelo en inglés no conoce `atención` ni `volveré`: los parte en trozos (`ate ##nc ##ion`, `vol ##vere`) y les quita las tildes, así que ninguno de esos trozos tiene un significado que el modelo haya aprendido. Con 15 frases tendría que aprender español desde cero, y eso no es posible.

El modelo en español reconoce `atención`, `volveré` y `nunca` como palabras completas. Ya sabe lo que significan; solo falta enseñarle a convertir eso en *positivo* o *negativo*.

## 2. Los datos

- **Entrenamiento**: 25 frases positivas y 25 negativas. Las primeras 10 positivas y 5 negativas son las 15 frases originales.
- **Prueba**: 10 + 10 frases **nuevas** que el modelo no ve al entrenar. Es la única forma honesta de saber si aprendió.

In [3]:
positivas = [
    "Me encanta este producto, funciona de maravilla.",
    "El servicio fue excelente y muy rápido.",
    "Estoy muy feliz con el resultado obtenido.",
    "La atención al cliente fue muy amable y eficiente.",
    "La película fue increíble, me hizo llorar de emoción.",
    "Definitivamente volvería a comprar aquí.",
    "El lugar es hermoso y muy acogedor.",
    "Mi experiencia fue fantástica, todo salió perfecto.",
    "La comida estuvo deliciosa, totalmente recomendada.",
    "Es una de las mejores decisiones que he tomado.",
    "El pedido llegó antes de lo esperado y en perfecto estado.",
    "Muy buena calidad por el precio que tiene.",
    "El hotel estaba limpio y el personal fue encantador.",
    "Superó todas mis expectativas, lo recomiendo.",
    "La batería dura muchísimo, estoy muy contento.",
    "Un libro maravilloso, no podía dejar de leerlo.",
    "El concierto fue espectacular, volvería sin pensarlo.",
    "Me resolvieron el problema en cinco minutos, genial.",
    "Las zapatillas son cómodas y se ven muy bien.",
    "Gran relación calidad precio, cinco estrellas.",
    "El curso me pareció muy útil y bien explicado.",
    "La app es intuitiva y funciona sin errores.",
    "Nos trataron como en casa, todo un acierto.",
    "Quedé encantada con el corte de pelo.",
    "El envío fue rapidísimo y el empaque impecable.",
]

negativas = [
    "El producto llegó roto y de mala calidad.",
    "La atención fue pésima, no volveré nunca.",
    "Me siento decepcionado, esperaba mucho más.",
    "La película fue aburrida y sin sentido.",
    "El servicio fue lento y los empleados groseros.",
    "No funciona, lo devolví al día siguiente.",
    "Una pérdida de dinero, no lo recomiendo.",
    "La comida estaba fría y sin sabor.",
    "El pedido nunca llegó y nadie me responde.",
    "La habitación estaba sucia y olía mal.",
    "Se rompió a la semana de usarlo.",
    "Muy caro para lo poco que ofrece.",
    "El peor restaurante en el que he estado.",
    "La batería se descarga en dos horas, un desastre.",
    "Me cobraron de más y no me quisieron devolver el dinero.",
    "El libro es tedioso y está mal escrito.",
    "Tuve que esperar una hora para que me atendieran.",
    "La talla no corresponde y la tela es horrible.",
    "La app se cierra sola todo el tiempo.",
    "Estoy muy molesto con la empresa, un fraude.",
    "El curso fue una pérdida de tiempo, muy desordenado.",
    "El vendedor fue maleducado y poco profesional.",
    "Las instrucciones son confusas y faltaban piezas.",
    "No cumple con lo que promete la publicidad.",
    "Llegó tarde, incompleto y mal embalado.",
]

prueba_positivas = [
    "Todo llegó bien y el producto es justo lo que buscaba.",
    "El mesero fue muy atento, volveremos pronto.",
    "Funciona perfecto, estoy satisfecho con la compra.",
    "Qué buena serie, la terminé en un fin de semana.",
    "Excelente trato del personal, muy recomendable.",
    "El teléfono es rápido y la cámara es buenísima.",
    "Me gustó mucho, cumple con todo lo prometido.",
    "La clase fue entretenida y aprendí bastante.",
    "No tuve ningún problema, todo perfecto.",
    "Lo recomiendo, vale cada sol que pagué.",
]

prueba_negativas = [
    "El producto vino defectuoso y no sirve para nada.",
    "Pésima experiencia, el personal no sabía nada.",
    "No me gustó, la calidad es muy mala.",
    "La serie es lenta y el final es terrible.",
    "Esperé tres semanas y me llegó otra cosa.",
    "El teléfono se calienta y se apaga solo.",
    "No vale lo que cuesta, me arrepiento de comprarlo.",
    "La clase fue un caos y el profesor no explicó nada.",
    "Horrible, jamás vuelvo a comprar en esta tienda.",
    "El sabor era desagradable y nos cobraron caro.",
]


def crear_dataset(textos_positivos, textos_negativos):
    return Dataset.from_dict({
        "text": textos_positivos + textos_negativos,
        "label": [1] * len(textos_positivos) + [0] * len(textos_negativos),
    })


datos_originales = crear_dataset(positivas[:10], negativas[:5])   # las 15 frases de la primera versión
datos_frases = crear_dataset(positivas, negativas)
prueba_frases = crear_dataset(prueba_positivas, prueba_negativas)

print(f"Original: {len(datos_originales)} | Balanceado: {len(datos_frases)} | Prueba: {len(prueba_frases)}")

Original: 15 | Balanceado: 50 | Prueba: 20


## 3. Funciones para entrenar y evaluar

Vamos a entrenar varios modelos, así que juntamos los pasos en funciones:

- `tokenizar`: corta en 128 tokens como máximo y **no rellena**. El `DataCollatorWithPadding` rellena cada lote solo hasta la frase más larga de ese lote.
- `calcular_metricas`: el `Trainer` la usa al evaluar para reportar *accuracy* y *F1*.
- `entrenar`: carga el modelo con una capa de clasificación nueva (2 salidas) y lo ajusta. Le pasamos `id2label` para que el modelo guardado responda `positivo`/`negativo` en lugar de `LABEL_0`/`LABEL_1`.

In [4]:
def tokenizar(dataset, tokenizer):
    return dataset.map(lambda lote: tokenizer(lote["text"], truncation=True, max_length=128), batched=True)


def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    predicciones = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(etiquetas, predicciones), "f1": f1_score(etiquetas, predicciones)}


def entrenar(nombre_modelo, datos_entrenamiento, epocas, batch_size=8, learning_rate=5e-5):
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    model = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=2,
        id2label=ETIQUETAS,
        label2id={etiqueta: id_ for id_, etiqueta in ETIQUETAS.items()},
    )

    training_args = TrainingArguments(
        output_dir=".results",
        num_train_epochs=epocas,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        warmup_steps=0.1,          # el 10% inicial de los pasos sube el learning rate poco a poco
        weight_decay=0.01,
        logging_strategy="epoch",
        save_strategy="no",        # sin checkpoints intermedios: guardamos solo el modelo final
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenizar(datos_entrenamiento, tokenizer),
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=calcular_metricas,
    )
    trainer.train()
    return trainer, tokenizer


def evaluar(trainer, tokenizer, datos_prueba):
    metricas = trainer.evaluate(tokenizar(datos_prueba, tokenizer))
    return {"accuracy": round(metricas["eval_accuracy"], 3), "f1": round(metricas["eval_f1"], 3)}

## 4. ¿Qué pesa más: el modelo o los datos?

Entrenamos las 4 combinaciones (inglés o español, con 15 o con 50 frases) durante 10 épocas y evaluamos con las 20 frases de prueba. Con 20 frases, un modelo que adivina al azar acierta en torno al 50%.

In [5]:
modelos = {}
resultados = []

for nombre_modelo in [MODELO_INGLES, MODELO_ESPANOL]:
    for nombre_datos, datos in [("15 frases (original)", datos_originales), ("50 frases balanceadas", datos_frases)]:
        trainer, tokenizer = entrenar(nombre_modelo, datos, epocas=10)
        modelos[(nombre_modelo, nombre_datos)] = (trainer, tokenizer)
        resultados.append({"modelo": nombre_modelo, "datos": nombre_datos,
                           **evaluar(trainer, tokenizer, prueba_frases)})

pd.DataFrame(resultados)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10186.28it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 15/15 [00:00<00:00, 4192.35 examples/s]
/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-pack

Step,Training Loss
2,0.702544
4,0.658318
6,0.612253
8,0.598656
10,0.521978
12,0.478352
14,0.428296
16,0.398027
18,0.329914
20,0.291181


Map: 100%|██████████| 20/20 [00:00<00:00, 6964.39 examples/s]


Training Loss,Validation Loss,Step,Accuracy,F1
0.291181,0.799705,20,0.500000,0.666667


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15515.50it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 50/50 [00:00<00:00, 18732.93 examples/s]
/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-pac

...

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 37749.11it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: dccuchile/distilbert-base-spanish-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 50/50 [00:00<00:00, 20576.45 examples/s]
/Users/marcell/Documents/cod

Step,Training Loss
7,0.705773
14,0.518410
21,0.276006
28,0.105326
35,0.037093
42,0.018868
49,0.013707
56,0.010143
63,0.009070
70,0.008704


Map: 100%|██████████| 20/20 [00:00<00:00, 9057.99 examples/s]


Training Loss,Validation Loss,Step,Accuracy,F1
0.008704,0.009367,70,1.000000,1.000000


,modelo,datos,accuracy,f1
0,distilbert-base-uncased,15 frases (original),0.50,0.667
1,distilbert-base-uncased,50 frases balanceadas,0.65,0.632
2,dccuchile/distilbert-base-spanish-uncased,15 frases (original),0.65,0.741
3,dccuchile/distilbert-base-spanish-uncased,50 frases balanceadas,1.00,1.000


- **Inglés + 15 frases (lo que teníamos): 50%**, lo mismo que tirar una moneda. El modelo no aprendió nada útil.
- Dar más frases al modelo en inglés ayuda poco (65%). Cambiar solo el modelo, manteniendo las 15 frases desbalanceadas, tampoco basta (65%).
- **Español + 50 frases balanceadas: 100%** en las 20 frases nuevas.

La diferencia la hace **combinar** un modelo base que entiende español con unos pocos ejemplos balanceados. Por separado, ninguna de las dos mejoras alcanza.

*Los números pueden variar un poco en cada ejecución, sobre todo en MPS o GPU.*

## 5. Prueba con reseñas reales

Nuestras 20 frases de prueba son cortas y claras, y las escribimos con el mismo estilo que las de entrenamiento, así que el resultado anterior es **optimista**. Para una prueba más exigente usamos las reseñas de Amazon en español del notebook 43 (`data/resenas_amazon_es.csv`):

- 1 y 2 estrellas son **negativas**; 4 y 5 estrellas, **positivas**.
- Quitamos las de 3 estrellas porque son ambiguas.
- Unimos título y texto, porque el título suele resumir la opinión.
- Separamos **1,000 reseñas de prueba**, que no se usan para entrenar.

In [6]:
df = pd.read_csv("../data/resenas_amazon_es.csv")
df = df[df["estrellas"] != 3].copy()
df["text"] = df["titulo"] + ". " + df["texto"]
df["label"] = (df["estrellas"] >= 4).astype(int)

resenas_entrenamiento, resenas_prueba = train_test_split(
    df[["text", "label"]], test_size=1000, stratify=df["label"], random_state=SEED
)
prueba_resenas = Dataset.from_pandas(resenas_prueba, preserve_index=False)

print(f"Reseñas disponibles para entrenar: {len(resenas_entrenamiento)} | prueba: {len(prueba_resenas)}")
print(resenas_prueba["label"].value_counts().rename(ETIQUETAS))
resenas_prueba.head()

Reseñas disponibles para entrenar: 7000 | prueba: 1000
label
positivo    500
negativo    500
Name: count, dtype: int64


,text,label
8230,UTIL. Funciona bien y no hemos tenido ningún p...,1
4006,Un sonido increíble.. He probado varios auricu...,1
784,Defectuoso. Tal como viene se va.. Descripción...,0
3064,Correcto. Todo bien. Mesa sencilla pero funcio...,1
8583,Buena compra. Muy fuerte parece de muy buena c...,1


In [7]:
resenas_entrenamiento.value_counts("label")

label
1    3500
0    3500
Name: count, dtype: int64

In [8]:
trainer_frases, tokenizer_frases = modelos[(MODELO_ESPANOL, "50 frases balanceadas")]
evaluar(trainer_frases, tokenizer_frases, prueba_resenas)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 57094.10 examples/s]
/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,F1
0.008704,0.465017,70,0.855000,0.842562


{'accuracy': 0.855, 'f1': 0.843}

Con solo 50 frases escritas a mano, el modelo acierta el **85.5%** de 1,000 reseñas reales que nunca vio, cuando el azar daría un 50%. Es un resultado decente.

El 100% anterior era optimista. Las reseñas reales son más largas, tienen faltas de ortografía, mezclan cosas buenas y malas, y hablan de envíos, devoluciones o vendedores, temas que nuestras frases casi no cubren.

## 6. Más datos reales: 2,000 reseñas

Usamos 2,000 reseñas de entrenamiento (alrededor de un minuto en MPS) y entrenamos solo 2 épocas: con miles de ejemplos, el modelo no necesita repasarlos 10 veces y así sobreajusta menos. Subimos el `batch_size` a 16 porque ahora hay datos de sobra.

In [9]:
datos_resenas = Dataset.from_pandas(resenas_entrenamiento.sample(7000, random_state=SEED), preserve_index=False)

trainer_resenas, tokenizer_resenas = entrenar(MODELO_ESPANOL, datos_resenas, epocas=2, batch_size=16)
modelos[(MODELO_ESPANOL, "2,000 reseñas")] = (trainer_resenas, tokenizer_resenas)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 32311.10it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: dccuchile/distilbert-base-spanish-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 7000/7000 [00:00<00:00, 78584.75 examples/s]
/Users/marcell/Documents

Step,Training Loss
438,0.269695
876,0.117664


## 7. Comparación final

Evaluamos todos los modelos con los dos conjuntos de prueba: nuestras 20 frases y las 1,000 reseñas reales.

In [10]:
comparacion = []
for (nombre_modelo, nombre_datos), (trainer, tokenizer) in modelos.items():
    comparacion.append({
        "modelo": nombre_modelo,
        "datos": nombre_datos,
        "accuracy 20 frases": evaluar(trainer, tokenizer, prueba_frases)["accuracy"],
        "accuracy 1,000 reseñas": evaluar(trainer, tokenizer, prueba_resenas)["accuracy"],
    })

pd.DataFrame(comparacion)

Map: 100%|██████████| 20/20 [00:00<00:00, 8186.40 examples/s]


Training Loss,Validation Loss,Step,Accuracy,F1
0.291181,0.799705,20,0.500000,0.666667


Map: 100%|██████████| 1000/1000 [00:00<00:00, 37609.66 examples/s]
/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,F1
0.291181,0.908815,20,0.503000,0.668003


Map: 100%|██████████| 20/20 [00:00<00:00, 7641.99 examples/s]
/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Map: 100%|██████████| 20/20 [00:00<00:00, 9207.12 examples/s]
/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,F1
0.117664,0.007624,876,1.000000,1.000000


Map: 100%|██████████| 1000/1000 [00:00<00:00, 53438.11 examples/s]
/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,F1
0.117664,0.223922,876,0.940000,0.941061


,modelo,datos,accuracy 20 frases,"accuracy 1,000 reseñas"
0,distilbert-base-uncased,15 frases (original),0.50,0.503
1,distilbert-base-uncased,50 frases balanceadas,0.65,0.636
2,dccuchile/distilbert-base-spanish-uncased,15 frases (original),0.65,0.520
3,dccuchile/distilbert-base-spanish-uncased,50 frases balanceadas,1.00,0.855
4,dccuchile/distilbert-base-spanish-uncased,"2,000 reseñas",1.00,0.940


- **Original (inglés + 15 frases)**: también acierta un 50% de las reseñas reales. Es un modelo que adivina.
- **Español + 15 frases**: 52% en reseñas. Con tan pocos ejemplos, y desbalanceados, ni el mejor modelo base generaliza.
- **Español + 50 frases**: 85.5%. Con poca data, el salto grande viene de elegir bien el modelo base y balancear las clases.
- **Español + 2,000 reseñas**: **93.4%**. Entrenar con datos del mismo tipo que vamos a clasificar suma 8 puntos más, y el modelo sigue acertando las 20 frases.

## 8. Guardar y usar el modelo

Guardamos el modelo entrenado con reseñas **junto con su tokenizador**, porque sin el tokenizador no podríamos preparar textos nuevos. Después lo cargamos con `pipeline`, que se encarga de tokenizar, ejecutar el modelo y aplicar softmax para devolver una probabilidad.

In [13]:
RUTA_MODELO = "../data/modelo_bert_finetuned"

trainer_resenas.save_model(RUTA_MODELO)
tokenizer_resenas.save_pretrained(RUTA_MODELO)

clasificador = pipeline("text-classification", model=RUTA_MODELO, device=DEVICE)


def predecir_texto(texto):
    resultado = clasificador(texto)[0]
    return resultado["label"], round(resultado["score"], 3)


for texto in ["Estoy muy feliz con el resultado obtenido",
              "El producto llegó roto y de mala calidad"]:
    etiqueta, probabilidad = predecir_texto(texto)
    print(f"{etiqueta:>8} ({probabilidad:.3f})  {texto}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5874.38it/s]


positivo (0.998)  Estoy muy feliz con el resultado obtenido
negativo (0.996)  El producto llegó roto y de mala calidad


Probemos frases más difíciles: negaciones, contrastes con "pero" e ironía.

In [14]:
frases_dificiles = [
    "No está nada mal, me sorprendió para bien.",
    "Pensé que sería malo, pero me encantó.",
    "El envío fue rápido, pero el producto es malísimo.",
    "No lo recomiendo para nada.",
    "Genial, se rompió al segundo día.",
]

for texto in frases_dificiles:
    etiqueta, probabilidad = predecir_texto(texto)
    print(f"{etiqueta:>8} ({probabilidad:.3f})  {texto}")

positivo (0.997)  No está nada mal, me sorprendió para bien.
negativo (0.887)  Pensé que sería malo, pero me encantó.
negativo (0.957)  El envío fue rápido, pero el producto es malísimo.
negativo (0.996)  No lo recomiendo para nada.
positivo (0.995)  Genial, se rompió al segundo día.


El modelo resuelve bien las negaciones ("No está nada mal", "No lo recomiendo para nada"), pero tiene límites:

- **"Pensé que sería malo, pero me encantó"** sale *negativo*. Lo que va después del "pero" es lo que cuenta, y el modelo se queda con "malo".
- **"Genial, se rompió al segundo día"** sale *positivo*. La ironía es de lo más difícil en análisis de sentimiento: el modelo ve "genial" y no capta la burla.
- **"El envío fue rápido, pero el producto es malísimo"** sale bien, pero con poca seguridad (0.57), porque mezcla una opinión buena y una mala.

Estos casos mejoran si se agregan al entrenamiento más ejemplos de contrastes e ironía, o si se entrena con todas las reseñas.

## Conclusiones

¿Se puede conseguir algo decente con poca data? **Sí**, con tres condiciones:

1. **Un modelo base que conozca el idioma.** Es el cambio más importante, porque el fine-tuning solo ajusta lo que el modelo ya sabe. Para español sirven `dccuchile/distilbert-base-spanish-uncased` (DistilBERT, rápido) o `dccuchile/bert-base-spanish-wwm-uncased` (BETO, más grande).
2. **Clases balanceadas y ejemplos variados.** Con 25 ejemplos por clase, sobre temas distintos, llegamos al 85% en reseñas reales.
3. **Evaluar con datos que el modelo no vio.** Si se evalúa con los datos de entrenamiento, los números siempre salen buenos y no dicen nada.

Con datos reales (2,000 reseñas y alrededor de un minuto de entrenamiento) llegamos al 93%. Para seguir mejorando se pueden usar las 7,000 reseñas disponibles, agregar ejemplos de contrastes e ironía o crear una clase *neutral* con las reseñas de 3 estrellas.